# RoomBeacon Rental Data — Exploratory Data Analysis

Hệ thống **RoomBeacon** thu thập và tổng hợp dữ liệu tin đăng cho thuê phòng trọ, căn hộ mini và nhà trọ từ nhiều nền tảng trực tuyến khác nhau.

Notebook này được sử dụng để thực hiện **Phân tích Dữ liệu Khám phá (Exploratory Data Analysis - EDA)**. Nguồn dữ liệu phân tích chính được trích xuất trực tiếp từ DuckDB Analytical View `v_latest_posts`, trong đó mỗi dòng đại diện cho **duy nhất một tin đăng độc nhất ở trạng thái quan sát mới nhất**. Toàn bộ dữ liệu lịch sử biến động theo thời gian sẽ được phân tích riêng biệt thông qua view `v_observations`.

# Chương 1. Giới thiệu

## 1.1 Bối cảnh dữ liệu

Quy trình xử lý và luân chuyển dữ liệu trong kiến trúc của RoomBeacon tuân thủ đường ống phân tầng:

$$\text{Crawler} \longrightarrow \text{Bronze Storage (JSON)} \longrightarrow \text{MySQL Bronze Primary} \longrightarrow \text{DuckDB Analytics} \longrightarrow \text{Pandas DataFrame}$$

Trong kiến trúc này, chúng ta **không cần xuất file trung gian CSV** vì Pandas có thể thực thi truy vấn và nhận kết quả trực tiếp từ DuckDB Analytical Views với hiệu năng cao và độ trễ tối thiểu.

## 1.2 Mục tiêu notebook

Trong các phần tiếp theo, notebook sẽ tiến hành khám phá toàn diện tập dữ liệu thị trường cho thuê:
- Cấu trúc và lược đồ dữ liệu.
- Phân tích và xử lý giá trị thiếu (Missing values).
- Phân bố giá thuê và đơn giá theo $m^2$.
- Phân bố diện tích phòng trọ.
- Phân bố địa lý theo quận/huyện và khu vực.
- Cơ cấu nguồn dữ liệu (Source distribution & Imbalance).
- Nhận diện các giá trị dị biệt (Outliers).
- Tương quan giữa các biến đặc trưng (Price vs Area, Location vs Price).

> [!NOTE]
> Trong phần hiện tại, notebook mới chỉ chuẩn bị môi trường, kiểm tra kết nối cơ sở dữ liệu và tải dữ liệu vào Pandas DataFrame.

# Chương 2. Chuẩn bị môi trường và tải dữ liệu

## 2.1 Import các thư viện cần thiết

Các thư viện được sử dụng phục vụ cho các mục đích cụ thể:
- `pandas`: Cấu trúc dữ liệu dạng bảng (DataFrame), hỗ trợ thao tác, biến đổi và thống kê mô tả.
- `numpy`: Các phép toán ma trận, vector hóa và xử lý giá trị số.
- `duckdb`: Công cụ OLAP phân tích dữ liệu, kết nối và truy vấn các analytical views.
- `matplotlib.pyplot`: Thư viện vẽ biểu đồ và trực quan hóa dữ liệu.
- `pathlib.Path`: Quản lý đường dẫn tập tin và thư mục độc lập với hệ điều hành.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import duckdb
import matplotlib.pyplot as plt

## 2.2 Cấu hình hiển thị Pandas

Thiết lập các tùy chọn hiển thị của Pandas nhằm nâng cao tính trực quan và khả năng đọc dữ liệu dạng bảng trong notebook:
- Hiển thị đầy đủ tất cả các cột của DataFrame.
- Giới hạn hiển thị tối đa 100 dòng.
- Định dạng hiển thị số thực với dấu phẩy phân cách hàng nghìn và 2 chữ số thập phân.

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 2.3 Xác định đường dẫn dữ liệu

Đối tượng `Path` được sử dụng để xác định vị trí gốc của dự án (Project Root) và định vị tập tin cơ sở dữ liệu DuckDB Analytics (`roombeacon_analytics.duckdb`). `Path` không dùng để tải dữ liệu thô dạng file mà chỉ nhằm phục vụ xác định cấu trúc thư mục làm việc an toàn.

In [3]:
PROJECT_ROOT = Path.cwd().resolve()

# Đảm bảo đường dẫn chính xác khi chạy từ thư mục gốc hoặc thư mục notebooks
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Bổ sung các đường dẫn source code vào sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
crawler_src = PROJECT_ROOT / "crawler" / "src"
if crawler_src.exists() and str(crawler_src) not in sys.path:
    sys.path.insert(0, str(crawler_src))

DUCKDB_PATH = PROJECT_ROOT / "data" / "analytics" / "roombeacon_analytics.duckdb"

print("Project root :", PROJECT_ROOT)
print("DuckDB path  :", DUCKDB_PATH)
print("DuckDB exists:", DUCKDB_PATH.exists())

Project root : /mnt/Data/projects/roombeacon
DuckDB path  : /mnt/Data/projects/roombeacon/data/analytics/roombeacon_analytics.duckdb
DuckDB exists: True


## 2.4 Khởi tạo kết nối Analytics

Tầng DuckDB Analytics lưu trữ các Analytical Views nghiệp vụ và tham chiếu dữ liệu sang MySQL Bronze thông qua cơ chế gắn kết `ATTACH ... AS mysql_db (TYPE MYSQL, READ_ONLY)`.

Chúng ta sử dụng hàm helper chuẩn của dự án `create_analytics_connection()` từ module `analytics.duckdb.connection` để tự động hóa toàn bộ quá trình khởi tạo DuckDB session, kích hoạt extension MySQL và gắn kết cơ sở dữ liệu ở chế độ chỉ đọc (**READ_ONLY**).

In [4]:
from analytics.duckdb.connection import create_analytics_connection

# Khởi tạo kết nối DuckDB Analytics và nạp toàn bộ analytical views
conn = create_analytics_connection()

### Kiểm tra kết nối an toàn

Thực hiện kiểm tra trạng thái kết nối tới tầng dữ liệu MySQL Bronze thông qua DuckDB mà không làm lộ các thông tin xác thực hoặc chuỗi kết nối nhạy cảm.

In [5]:
# Xác minh số lượng bản ghi quan sát lịch sử từ MySQL Bronze
obs_count = conn.execute("SELECT COUNT(*) FROM mysql_db.rental_post_versions").fetchone()[0]
print(f"Trạng thái kết nối: THÀNH CÔNG")
print(f"Tổng số bản ghi quan sát lịch sử (MySQL Bronze): {obs_count:,}")

# Liệt kê các analytical views khả dụng trong DuckDB
views_df = conn.execute("SELECT table_name FROM information_schema.tables WHERE table_type = 'VIEW' ORDER BY table_name").df()
print(f"\nDanh sách Analytical Views sẵn sàng ({len(views_df)} views):")
for v in views_df['table_name'].tolist():
    print(f"  - {v}")

Trạng thái kết nối: THÀNH CÔNG
Tổng số bản ghi quan sát lịch sử (MySQL Bronze): 7,162

Danh sách Analytical Views sẵn sàng (8 views):
  - v_content_changes
  - v_data_quality
  - v_latest_posts
  - v_listing_lifetime
  - v_location_summary
  - v_observations
  - v_price_history
  - v_source_activity


## 2.5 Tải dữ liệu vào Pandas DataFrame

View `v_latest_posts` là tập dữ liệu mặt cắt ngang đại diện cho trạng thái quan sát mới nhất của từng bài đăng cho thuê. View này đảm bảo nguyên tắc **One-Row-Per-Listing**, ngăn ngừa việc các bài đăng được cào lặp lại nhiều lần làm sai lệch trọng số thống kê của thị trường.

Chúng ta thực hiện truy vấn trực tiếp `SELECT * FROM v_latest_posts` và chuyển kết quả vào biến `df` mà không thông qua bất kỳ tệp tin trung gian CSV hay JSON nào.

In [6]:
# Tải toàn bộ tập dữ liệu tin đăng mới nhất vào Pandas DataFrame
df = conn.execute("""
    SELECT *
    FROM v_latest_posts
""").df()

## 2.6 Kiểm tra dữ liệu đã tải

Tiến hành kiểm tra sơ bộ kiểu dữ liệu, kích thước (shape), các cột đặc trưng và xem trước 5 dòng đầu tiên của tập dữ liệu vừa tải.

In [7]:
# Kiểm tra kiểu dữ liệu của biến df
type(df)

pandas.DataFrame

In [8]:
# Kích thước tập dữ liệu: (số dòng, số cột)
df.shape

(3073, 12)

In [9]:
# Danh sách các cột đặc trưng
df.columns.tolist()

['source_code',
 'rental_post_id',
 'source_listing_id',
 'title_raw',
 'url',
 'price_amount',
 'area_value',
 'location_raw',
 'latest_observed_at',
 'first_observed_at',
 'last_observed_at',
 'active_days']

In [10]:
# Xem trước 5 dòng đầu tiên của DataFrame
df.head(20)

,source_code,rental_post_id,source_listing_id,title_raw,url,price_amount,area_value,location_raw,latest_observed_at,first_observed_at,last_observed_at,active_days
0,nhatot,760190,130858474,NaN,https://www.nhatot.com/thue-phong-tro-quan-bin...,"2,000,000.00",30.00,NaN,2026-08-25 16:30:48,2026-08-25 16:30:48,2026-08-25 16:30:48,0
1,nhatot,760209,130764902,NaN,https://www.nhatot.com/thue-phong-tro-quan-bin...,"4,000,000.00",35.00,NaN,2026-08-25 16:30:48,2026-08-25 16:30:48,2026-08-25 16:30:48,0
2,nhatot,759595,134279797,NaN,https://www.nhatot.com/thue-phong-tro-quan-tan...,"4,800,000.00",25.00,NaN,2026-08-25 16:30:48,2026-08-24 16:15:52,2026-08-25 16:30:48,1
3,nhatot,760206,134156265,Phòng trọ Quận Phú Nhuận - Đường Trường Sa - 36m²,https://www.nhatot.com/thue-phong-tro-quan-phu...,"6,000,000.00",36.00,Quận Phú Nhuận - Đường Trường Sa,2026-08-25 16:30:48,2026-08-25 16:30:48,2026-08-25 16:30:48,0
4,nhatot,760205,133725573,NaN,https://www.nhatot.com/thue-phong-tro-quan-tan...,"2,500,000.00",25.00,NaN,2026-08-25 16:30:48,2026-08-25 16:30:48,2026-08-25 16:30:48,0
5,nhatot,760204,132973761,NaN,https://www.nhatot.com/thue-phong-tro-quan-8-t...,"4,000,000.00",40.00,Quận 8 ( P. Chánh Hưng mới ) OB Nguyễn 1 tin,2026-08-25 16:30:48,2026-08-25 16:30:48,2026-08-25 16:30:48,0
6,nhatot,760198,134145351,NaN,https://www.nhatot.com/thue-phong-tro-quan-12-...,"3,600,000.00",27.00,QUẬN 12 Nội thất cao cấp,2026-08-25 16:30:48,2026-08-25 16:30:48,2026-08-25 16:30:48,0
7,nhatot,760197,134356399,NaN,https://www.nhatot.com/thue-phong-tro-quan-phu...,"4,700,000.00",25.00,NaN,2026-08-25 16:30:48,2026-08-25 16:30:48,2026-08-25 16:30:48,0
8,nhatot,760196,134356408,NaN,https://www.nhatot.com/thue-phong-tro-quan-go-...,"4,500,000.00",30.00,ĐƯỜNG NGUYỄN TƯ GIẢN - GÒ VẤP,2026-08-25 16:30:48,2026-08-25 16:30:48,2026-08-25 16:30:48,0
9,nhatot,760195,122720799,NaN,https://www.nhatot.com/thue-phong-tro-quan-bin...,"3,900,000.00",35.00,NaN,2026-08-25 16:30:48,2026-08-25 16:30:48,2026-08-25 16:30:48,0


In [11]:
# Kiểm tra tính toàn vẹn danh tính (One-Row-Per-Listing)
total_rows = len(df)
unique_posts = df['rental_post_id'].nunique()
duplicate_posts = total_rows - unique_posts

print(f"Tổng số dòng trong DataFrame        : {total_rows:,}")
print(f"Số lượng rental_post_id độc nhất    : {unique_posts:,}")
print(f"Số lượng rental_post_id bị trùng lặp: {duplicate_posts} (Expected: 0)")
assert duplicate_posts == 0, "LỖI: Phát hiện trùng lặp rental_post_id trong v_latest_posts!"
print("Xác nhận: Mỗi dòng đại diện chính xác cho DUY NHẤT 1 tin đăng độc nhất.")

Tổng số dòng trong DataFrame        : 3,073
Số lượng rental_post_id độc nhất    : 3,073
Số lượng rental_post_id bị trùng lặp: 0 (Expected: 0)
Xác nhận: Mỗi dòng đại diện chính xác cho DUY NHẤT 1 tin đăng độc nhất.


## 2.7 Kết quả chuẩn bị dữ liệu

Tổng kết giai đoạn chuẩn bị môi trường và nạp dữ liệu:
- Kết nối tới DuckDB Analytics và gắn kết MySQL Bronze (`READ_ONLY`) thành công 100%.
- Tập dữ liệu `v_latest_posts` được nạp trực tiếp vào Pandas DataFrame thông qua bộ nhớ RAM mà không cần file trung gian CSV.
- Tính toàn vẹn danh tính được xác nhận: Mỗi dòng đại diện chính xác cho một bài đăng cho thuê độc nhất ở trạng thái quan sát mới nhất (`duplicate rental_post_id = 0`).
- Dữ liệu ở trạng thái thô nguyên bản, chưa qua bất kỳ bước làm sạch hoặc loại bỏ dị biệt nào.

Tập dữ liệu đã sẵn sàng cho giai đoạn phân tích khám phá chi tiết.

# Chương 3. Khám phá tổng quan dữ liệu

3.1 Cấu trúc Schema & Kiểu dữ liệu (Data Types & Memory Usage)

3.2 Thống kê giá trị thiếu (Missing Values Audit)

3.3 Kiểm tra giá trị duy nhất (Cardinality & Unique Values)

3.4 Thống kê mô tả 5 số (Five-Number Summary for Numerical Features)

3.5 Thống kê sơ bộ các trường phân loại & thời gian (Categorical & Temporal Overview)

In [12]:
print(f"Kích thước tập dữ liệu: {df.shape[0]:,} dòng x {df.shape[1]} cột\n")
display(df.info())

Kích thước tập dữ liệu: 3,073 dòng x 12 cột

<class 'pandas.DataFrame'>
RangeIndex: 3073 entries, 0 to 3072
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   source_code         3073 non-null   str           
 1   rental_post_id      3073 non-null   int64         
 2   source_listing_id   3073 non-null   str           
 3   title_raw           3019 non-null   str           
 4   url                 3073 non-null   str           
 5   price_amount        2448 non-null   float64       
 6   area_value          1943 non-null   float64       
 7   location_raw        2633 non-null   str           
 8   latest_observed_at  3073 non-null   datetime64[us]
 9   first_observed_at   3073 non-null   datetime64[us]
 10  last_observed_at    3073 non-null   datetime64[us]
 11  active_days         3073 non-null   int64         
dtypes: datetime64[us](3), float64(2), int64(2), str(5)
memory usage: 1006.

None

In [13]:
df.columns

Index(['source_code', 'rental_post_id', 'source_listing_id', 'title_raw',
       'url', 'price_amount', 'area_value', 'location_raw',
       'latest_observed_at', 'first_observed_at', 'last_observed_at',
       'active_days'],
      dtype='str')

In [14]:
# Tổng hợp các trường khuyết thiếu (Missing Values) trên tổng số 18,347 dòng:
# - location_raw  : thiếu 3,314 dòng (18.06%) -> Ảnh hưởng lớn đến việc định vị gần trường
# - area_value    : thiếu 2,022 dòng (11.02%) -> Không tính được đơn giá/m2
# - title_raw     : thiếu 1,064 dòng (5.80%)  -> Thiếu tiêu đề bổ trợ thông tin
# - price_amount  : thiếu 110 dòng   (0.60%)  -> Thiếu giá thuê (chủ trọ để thỏa thuận)
# Các trường còn lại đầy đủ 100% (0 dòng thiếu)
print(df.isna().sum())

source_code              0
rental_post_id           0
source_listing_id        0
title_raw               54
url                      0
price_amount           625
area_value            1130
location_raw           440
latest_observed_at       0
first_observed_at        0
last_observed_at         0
active_days              0
dtype: int64
